# Нейронные сети и обработка естественного языка - NLP

# Модуль 8. AI-агенты

Для работы в рамках данного ноутбука понадобится GPU.



### HF-токен

**ВНИМАНИЕ**: Вам понадобится `HF_TOKEN` для загрузки некоторых моделей и использования API.

Как получить токен:

1. Зарегистрироваться на Hugging Face: https://huggingface.co/join
2. В меню по клику на аватар выбрать пункт [Access Tokens](https://huggingface.co/settings/tokens)

3. Нажать `New token`
4. Выбрать:

   * `read` — достаточно почти для всех курсов и скачивания моделей
5. Скопировать токен вида:

    ```text
    hf_xxxxxxxxxxxxxxxxx
    ```

Полученный токен следует разместить в файле ```__config__.py``` в виде переменной `HF_TOKEN`:

```python
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxx"
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import torch

from __config__ import *

from datasets import load_dataset

print(len(HF_TOKEN))


os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "./hf_cache" # для колаба ок

## Задача: разобрать папку Downloads

Загрузим прототип папки Downloads, которая есть на каждом компьютере. В ней может быть огромное количество файлов, и часто бывает сложно найти нужный. Попробуем решить эту проблему с помощью AI-агента.

Загрузка папки с GitHub и распковка:

In [ ]:
!mkdir -p data

!wget -O Downloads.zip \
https://github.com/easyise/spec_python_courses/raw/refs/heads/master/neural_02_nlp/data/Downloads.zip

!unzip -o Downloads.zip -d ./data/

In [ ]:
!pip install -q "smolagents[toolkit]" huggingface_hub pandas pillow imagehash opencv-python

Директория на растерзание агенту:

In [ ]:
ROOT = "./data/Downloads"

In [ ]:
import os
import json
import hashlib
import sqlite3
import zipfile
import mimetypes
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image
import imagehash
import cv2

from smolagents import CodeAgent, InferenceClientModel, tool

### AI Agent's Tools

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
@tool
def scan_folder(root: str) -> str:
    """
    Recursively scans a folder.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with information about discovered files.
    """
    root_path = Path(root)
    result = []

    for p in root_path.rglob("*"):
        if p.is_file():
            try:
                stat = p.stat()
                mime, _ = mimetypes.guess_type(str(p))

                result.append({
                    "path": str(p),
                    "name": p.name,
                    "ext": p.suffix.lower(),
                    "size_mb": round(stat.st_size / 1024 / 1024, 3),
                    "mime": mime,
                })

            except Exception as e:
                result.append({
                    "path": str(p),
                    "error": str(e),
                })

    return json.dumps(result, ensure_ascii=False, indent=2)

In [ ]:
@tool
def find_exact_duplicates(root: str) -> str:
    """
    Finds exact file duplicates by SHA-256 hash.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with SHA-256 hashes as keys and lists of duplicate file paths as values.
    """
    hashes = defaultdict(list)

    for p in Path(root).rglob("*"):
        if p.is_file():
            try:
                hashes[sha256_file(p)].append(str(p))
            except Exception:
                pass

    duplicates = {
        h: paths
        for h, paths in hashes.items()
        if len(paths) > 1
    }

    return json.dumps(duplicates, ensure_ascii=False, indent=2)

In [ ]:
@tool
def inspect_csv_files(root: str) -> str:
    """
    Inspects CSV files in a folder.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with CSV columns, data types, row counts and missing values.
    """
    result = []

    for p in Path(root).rglob("*.csv"):
        try:
            df = pd.read_csv(p, nrows=5000)
            result.append({
                "path": str(p),
                "columns": list(df.columns),
                "sample_rows_read": len(df),
                "missing_values": df.isna().sum().to_dict(),
                "dtypes": df.dtypes.astype(str).to_dict(),
            })
        except Exception as e:
            result.append({
                "path": str(p),
                "error": str(e),
            })

    return json.dumps(result, ensure_ascii=False, indent=2)

In [ ]:
@tool
def inspect_db3_files(root: str) -> str:
    """
    Inspects SQLite DB3 files in a folder.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with DB3 tables, columns and row counts.
    """
    result = []

    for p in Path(root).rglob("*.db3"):
        info = {"path": str(p), "tables": []}

        try:
            conn = sqlite3.connect(p)
            cur = conn.cursor()

            cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
            tables = [x[0] for x in cur.fetchall()]

            for table in tables:
                cur.execute(f"PRAGMA table_info({table})")
                columns = [row[1] for row in cur.fetchall()]

                try:
                    cur.execute(f"SELECT COUNT(*) FROM {table}")
                    row_count = cur.fetchone()[0]
                except Exception:
                    row_count = None

                info["tables"].append({
                    "table": table,
                    "columns": columns,
                    "row_count": row_count,
                })

            conn.close()

        except Exception as e:
            info["error"] = str(e)

        result.append(info)

    return json.dumps(result, ensure_ascii=False, indent=2)

In [ ]:
@tool
def inspect_video_files(root: str) -> str:
    """
    Inspects video files in a folder.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with video resolution, FPS, frame count and duration.
    """
    video_exts = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
    result = []

    for p in Path(root).rglob("*"):
        if p.suffix.lower() in video_exts and p.is_file():
            cap = cv2.VideoCapture(str(p))

            if not cap.isOpened():
                result.append({
                    "path": str(p),
                    "error": "Could not open video",
                })
                continue

            fps = cap.get(cv2.CAP_PROP_FPS)
            frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
            width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
            height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)

            duration = frames / fps if fps and fps > 0 else None

            result.append({
                "path": str(p),
                "width": int(width),
                "height": int(height),
                "fps": round(fps, 2) if fps else None,
                "frames": int(frames),
                "duration_sec": round(duration, 2) if duration else None,
            })

            cap.release()

    return json.dumps(result, ensure_ascii=False, indent=2)

In [ ]:
@tool
def inspect_archives(root: str) -> str:
    """
    Inspects ZIP archives in a folder.

    Args:
        root: Path to the folder that should be scanned.

    Returns:
        JSON string with archive contents and encrypted file indicators.
    """
    result = []

    for p in Path(root).rglob("*.zip"):
        info = {
            "path": str(p),
            "encrypted_files": [],
            "files": [],
        }

        try:
            with zipfile.ZipFile(p) as z:
                for item in z.infolist():
                    encrypted = bool(item.flag_bits & 0x1)

                    info["files"].append({
                        "name": item.filename,
                        "size_mb": round(item.file_size / 1024 / 1024, 3),
                        "encrypted": encrypted,
                    })

                    if encrypted:
                        info["encrypted_files"].append(item.filename)

        except Exception as e:
            info["error"] = str(e)

        result.append(info)

    return json.dumps(result, ensure_ascii=False, indent=2)

### Создаем агента

Создаем CodeAgent, который работает, генерируя код на Python для выполнения задач. У агента будет набор инструментов, которые он может использовать для анализа папки.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
model = InferenceClientModel(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    temperature=0.1,
    max_tokens=2000,
)

In [ ]:
agent = CodeAgent(
    tools=[
        scan_folder,
        find_exact_duplicates,
        inspect_csv_files,
        inspect_db3_files,
        inspect_video_files,
        inspect_archives,
    ],
    model=model,
    max_steps=8,
)

### Запуск агента

In [ ]:
from IPython.display import Markdown, display

task = f"""
Ты read-only агент-аудитор файлового хранилища.

Проанализируй папку:

{ROOT}

Нужно:
1. Кратко описать содержимое папки.
2. Найти точные дубликаты файлов.
3. Описать CSV-файлы: колонки, типы, пропуски.
4. Описать DB3-файлы: таблицы, колонки, количество строк.
5. Описать видео: разрешение, FPS, длительность.
6. Найти архивы и отметить, какие из них запаролены.
7. Составить аккуратный Markdown-отчёт.
8. Ничего не удалять и не изменять.
"""

report = agent.run(task)
Markdown(report)

#### Использование ToolCallingAgent

In [ ]:
from smolagents import ToolCallingAgent

agent = ToolCallingAgent(
    tools=[
        scan_folder,
        find_exact_duplicates,
        inspect_csv_files,
        inspect_db3_files,
        inspect_video_files,
        inspect_archives,
    ],
    model=model,
    max_steps=8,
)

report = agent.run(task)
Markdown(report)

**ПРАКТИКА**

Реализуйте следующие задачи для агента:
1. Составьте список самых больших файлов (по размеру), чтобы он включал их размеры и пути к ним.
2. Пусть напишет о чем текстовые файлы, которые он нашел.
3. Пусть найдет попробует догадаться, что за данные в файлах CSV и DB3.

In [ ]:
# ваш код здесь



